In [2]:
import cv2
import PIL
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from random import sample
from matplotlib.pyplot import figure
from sklearn.model_selection import StratifiedKFold
%matplotlib inline

### Images

In [3]:
def plotRawImages(label):
    size, rows, cols = 64, 5, 5
    data = pd.read_csv('../../data/raw/train.csv')
    data = data[data['label'] == label]
    data = sample(data['image_id'].tolist(), 25)
    path = '../../data/raw/train_images/'
    data = [path + x for x in data]
    mosaic = PIL.Image.new(mode='RGB', size=(size*cols + (cols-1), size*rows + (rows-1)))
    for idx, name in enumerate(data):
        img = cv2.imread(name, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ix  = idx % cols
        iy  = idx // cols
        img = np.clip(img, 0, 255).astype(np.uint8)
        img = PIL.Image.fromarray(img)
        img = img.resize((size, size), resample=PIL.Image.BILINEAR)
        mosaic.paste(img, (ix*size + ix, iy*size + iy))
    plt.figure(figsize=(12,12))
    plt.imshow(mosaic)
    return None

In [4]:
# plotRawImages(0)

In [5]:
# plotRawImages(1)

In [6]:
# plotRawImages(2)

In [7]:
# plotRawImages(3)

### Size

In [7]:
def getSize(image):
    path = '../../data/raw/train_images/'
    image = path + image
    image = cv2.imread(image, cv2.IMREAD_COLOR)
    height, width, _ = image.shape
    return (height, width)

In [8]:
data = pd.read_csv('../../data/raw/train.csv')
data['shape'] = data['image_id'].map(lambda x : getSize(x))
data['height'] = data['shape'].map(lambda x : x[0])
data['width'] = data['shape'].map(lambda x : x[1])

In [9]:
data.height.value_counts()

600    21397
Name: height, dtype: int64

In [10]:
data.width.value_counts()

800    21397
Name: width, dtype: int64

### Split

In [8]:
data = pd.read_csv('../../data/raw/train.csv')

In [9]:
split = StratifiedKFold(n_splits=5, random_state=2017, shuffle=True)

In [10]:
data['fold'] = -1

In [11]:
counter = 0
for train_idx, val_idx in split.split(data['image_id'], data['label']):
    data.loc[val_idx,'fold'] = counter
    counter += 1

In [12]:
data['fold'].value_counts()

1    4280
0    4280
4    4279
3    4279
2    4279
Name: fold, dtype: int64

In [13]:
data.label.value_counts()

3    13158
4     2577
2     2386
1     2189
0     1087
Name: label, dtype: int64

In [14]:
data.pivot_table(index='fold', columns='label', values='image_id', aggfunc='count')

label,0,1,2,3,4
fold,,,,,
0,218,438,477,2631,516
1,218,438,477,2631,516
2,217,438,477,2632,515
3,217,438,477,2632,515
4,217,437,478,2632,515


In [15]:
data.to_csv('../../data/raw/data.csv', index=False)

In [16]:
data.head()

,image_id,label,fold
0,1000015157.jpg,0,2
1,1000201771.jpg,3,3
2,100042118.jpg,1,2
3,1000723321.jpg,1,0
4,1000812911.jpg,3,0


In [17]:
data.shape

(21397, 3)

In [18]:
data['image_id'].nunique()

21397